# 4b · MIL training — instance-level, the instrument for Q2

> **The criterion in §2 was fixed on 13.08.2026, before any model existed, and nothing here has been
> run.** That ordering is the point: a structured-looking result read after the fact is not evidence
> ([TODO](../docs/TODO.md), *Agreed plan, Step 2*). §2 is now closed — every bar in it is a permutation
> null, a comparison against `4a`, or a collapse test, and no magnitude in it was chosen by judgement.
> The cells below implement it and may not be extended with a stage that is not in it.

## What this notebook is for

**Q2 — does within-line transcriptional heterogeneity survive into a model's per-cell response
predictions?** Scoped this way by Selin, 13.08.2026.

Concretely: do cells of one cell line differ enough in their representation — `pca` or `scgpt` — for a
model to assign them different response values, reproducibly and not as an artifact? Kinker et al.
(2020) documented recurrent heterogeneity programs in this dataset, but whether that heterogeneity
survives *our* preprocessing and embedding is **not assumed** — stage 0 measures it, and stage 0 can
fail.

It is the question the project is built around, because relapse is driven by rare surviving
subpopulations rather than by the average cell.

It is **structurally unanswerable** under `4a_percell_training`'s per-cell model: every cell of a line
carries that line's label, so the objective penalises exactly the within-line variation Q2 asks about
([Step 03](../docs/steps/03-model-and-training-design.md#every-cell-of-a-line-carries-the-identical-label)).
MIL is the smallest change that makes the question askable: a **bag of cells → one line label**
constrains only the aggregate, leaving the model free to differ between cells of the same line.

**The representation is part of the question, not a setting (Selin, 13.08.2026).** Q2 is asked of
`pca` and of `scgpt` separately and both arms run. One representation carrying within-line structure
while the other does not is a **result**, not a nuisance — it locates the structure in the encoding
rather than in the model.

### What Q2 claims here, and what it does not

Three questions had been running together under one name. Separated 13.08.2026:

| | | reachable with SCP542 + CTRPv2 |
|---|---|---|
| **a** | do per-cell predictions vary within a line, reproducibly, and not as a sequencing artifact? | **yes** — stages 0, 1, 2 and 6 below |
| **b** | is that variation *real* cellular heterogeneity of drug response? | **no** |
| **c** | does it predict *which* cells survive treatment? | **no** |

**(b) and (c) are out of reach for want of measurements, not because of this design.** No per-cell
response was ever measured — every label is one number per (cell line, drug) — and none of the four
primary sources carries post-treatment single-cell data. No threshold and no architecture closes that
gap; it takes a different dataset.

**So this notebook answers (a), and only (a).** That is worth having on its own terms: (a) is a
**necessary condition** for (b), so a model failing (a) has definitively not learned heterogeneity.
§3's program analysis is a **hint toward** (b) and explicitly not evidence for it. The write-up states
(b) and (c) as limitations of the data rather than leaving the narrowing implicit.

⚠️ **Candidate routes to (b) are recorded in
[Step 01](../docs/steps/01-datasets-and-harmonization.md#post-treatment-single-cell-data--what-would-be-needed-for-q2-b-and-what-exists)**
— MIX-seq, the palbociclib file already on disk, and lineage-barcoding studies, each with what it can
and cannot establish. **None is scoped, and none belongs to this notebook**; they are written down so
they are not rediscovered here as shortcuts.

## Decisions already taken

**Instance-level, not attention pooling (Selin, 12.08.2026).** Every cell gets its own *predicted
response*, and the line prediction aggregates them — rather than every cell getting an attention
*weight* over a pooled embedding. The two are the standard MIL alternatives (Ilse, Tomczak & Welling,
*Attention-based Deep Multiple Instance Learning*, ICML 2018): embedding-level usually predicts better,
instance-level is readable at the level of the individual instance. Q2 is a question about readability,
so the trade is taken deliberately and the cost in predictive performance is expected.

⚠️ **One consequence, recorded so it is not rediscovered:** selecting "the top-k cells" *by their
predicted value* and scoring that subset against the line's true response is biased by construction —
the extremes are shifted away from the line mean because they were chosen for being extreme. The
subpopulation-predictivity test that would have used it is therefore **not** in the criterion below.
It becomes available only if an attention weight is added alongside the per-cell predictions.

**Mean pooling, full-line bags, and the aggregator is never revisited (Selin, 13.08.2026).** The bag
prediction is the mean of its cells' predicted responses, over **all** of that line's cells. It matches
the synthetic control's own construction — a mixture-weighted *average* label — and it is fixed now,
before the model exists.

**Bag size is not a batching parameter.** For a sub-bag of `B` cells from a line of `n`, the expected
loss is `(p̄_n − y)² + (σ²/B)·(1 − (B−1)/(n−1))`: at `B = 1` that is exactly `4a`'s loss, at `B = n` the
variance term vanishes. Bag size therefore dials continuously between the per-cell model and MIL, and
sets how hard the objective charges for the very quantity Q2 measures. Full-line bags delete the term
outright — which is what stage 1's comparison against `4a` tests — and they are also what keeps the
[depth-weighting defect](../docs/steps/03-model-and-training-design.md#-open-defect--the-loss-weights-cell-lines-by-how-deeply-they-were-sequenced)
resolved, since under fixed-size sub-bags a deeply sequenced line yields proportionally more bags and
the depth weighting returns. Cost: one gradient step per line, and memory scaling with the largest
line.

⚠️ **Its cost, taken deliberately.** Mean pooling under an MSE-family loss carries a real shrinkage
incentive: collapsing every cell onto the bag mean is a minimiser whenever the model cannot do better,
and nothing in the objective rewards spreading cells apart. Both alternatives were considered and
rejected — a **variance term in the loss** would make stage 1 pass by construction and would change the
loss, which [the governing rule](../docs/TODO.md) forbids; a **max or top-quantile aggregator** does not
force variation either (the max of equal per-cell values is that value) and mismatches an averaged
label.

**The retry is ruled out, and that is the part that binds.** If stage 7 fails, the aggregator is *not*
swapped for one that passes. *Fail → change the instrument → retry* is a forking path moved down one
level, and it would leave any subsequent positive unattributable. A stage-7 failure **ends the run**,
and what it means is fixed here rather than after the numbers are seen: **Q2 unanswered, the instrument
was not demonstrated** — never "no heterogeneity found".

**Same scorer as the per-cell model.** This notebook writes out-of-fold predictions in the shared
format — one row per cell line × drug × arm — so [`5_evaluation`](5_evaluation.ipynb) computes order,
top-of-order, values and spread for MIL and the per-cell model through identical code. The two are
comparable because they went through the same scorer, not because two notebooks agree by convention.

**Loss:** whatever `4a_percell_training` settles on, unchanged, so the architecture is the only thing
that moves ([the governing rule](../docs/TODO.md)). Ranking losses (RankNet, LambdaRank) become
well-posed *here* and nowhere earlier — they need one score per cell line, which is what a bag produces
— but they are a second change and belong to a later run, not this one.


## 2 · What counts as a positive Q2 result — fixed before the run

**Settled by Selin, 13.08.2026. Every bar below is a permutation null, a comparison against
`4a_percell_training`, or a collapse test — the criterion contains no magnitude chosen by judgement.**

That was not true of the first draft, which carried three invented numbers: a floor on stage 0, an
AUROC bar on stage 7, and a fraction on stage 2. Each was removed by **replacing the bar with a null or
a comparison**, not by picking a better number. The one that mattered was stage 7's: a stage-7 failure
ends the run, so an arbitrary bar there decided whether the project reported anything at all.

**There is no ground truth for within-line heterogeneity of drug response.** Every label is one number
per (cell line, drug); no per-cell response was ever measured, and SCP542 carries no post-treatment
single-cell data, so the ideal test — do the model's resistant cells match the cells that actually
survive treatment — cannot be run here. That is question **(b)** in §1, out of reach for want of
measurements.

What can be established is narrower and still worth having: **that the model's per-cell predictions
vary within a line, that the variation is reproducible rather than noise, and that it is not a
sequencing artifact.**

Five stages, in this order. Two are preconditions, one is a necessary condition, one is the test, one
is a veto.

| # | Stage | Role | Passes when |
|---|---|---|---|
| **0** | **Input ceiling** — within-line dispersion of the cell representations, per representation, **before any training** | precondition on the **input** | the cells of a line are not collapsed to a point (§2.1) |
| **7** | **Synthetic positive control** — bags mixed from two lines of known, slightly different response; scored **inside the bag**, source-A cells against source-B cells | precondition on the **instrument** | the within-bag rank separation beats its permutation null (§2.2) |
| **1** | **Spread** — within-line standard deviation of per-cell predicted responses | necessary condition | it exceeds `4a`'s, on the same lines and drugs (§2.3) |
| **2** | **Reproducibility** — do independent seeds assign high and low predictions to the *same* cells? | **the test** | per-cell cross-seed agreement beats the shuffled-cell null (§2.4) |
| **6** | **Confound regression** — per-cell predictions against total counts, genes detected, mitochondrial fraction and cell-cycle score | **veto** | the confounds do *not* explain the variation (§2.5) |

**Why stage 0 comes first.** It costs no training. If the cells of a line collapse to a point in the
representation, no model of any kind can assign them different values, and every later stage is
measuring the wrong thing. It also separates two findings that a training run alone conflates: *the
input carries no within-line structure* and *the model did not use the structure that was there* — the
first is a statement about the representation, the second about the model. Because Q2 is asked of `pca`
and `scgpt` separately, stage 0 is per-representation and **may pass for one and fail for the other**,
which is itself a result.

**Why stage 7 comes before the rest.** Without it a negative is uninterpretable — "no heterogeneity
found" cannot be distinguished from "this method cannot find heterogeneity". With it, a negative becomes
a result: no detectable heterogeneity, by an instrument shown to detect it when present, **at a measured
sensitivity that is reported alongside the negative.**

**Why stage 7 is scored inside the bag and not on the bag (Selin, 13.08.2026).** The bag prediction is
the mean of its cells' predictions, so a model that assigns **every cell in the bag an identical value**
can land the bag mean exactly right and pass a bag-level test — while doing none of what stages 1, 2
and 6 go on to measure. Bag-level recovery would certify a shrunk instrument. It is still computed
(predicted bag value against mixture weight, over weights 0 … 1 in steps of 0.25) and **reported as a
diagnostic; it cannot pass the stage.**

**Why stage 6 is a veto and not an analysis.** It looks descriptive, but it can turn a pass into a
fail: predictions that replicate across seeds *and* are explained by library size are a sequencing
artifact, not biology. Pre-registered here so it cannot become something run only when the answer is
unwelcome.

### 2.1 · Stage 0 — the input ceiling, and why it has no floor

**Statistic:** per representation, the **within-line share of total variance** — mean over lines of the
within-line variance of the cell embeddings, divided by the total variance over all cells. It is one
minus the intraclass correlation, an ordinary quantity, and being a ratio it is scale-free, so `pca`
and `scgpt` are on one axis despite sharing no units.

**It is reported, and it fails only on collapse.** An earlier draft put a floor at 0.10. That number
was dropped on 13.08.2026 (Selin) for two reasons: it had no source, and it was *stricter than the
precondition it stood for*. Stage 0 exists to rule out one thing — that the cells of a line are
numerically indistinguishable, so that no model of any kind could separate them. That is the bar.
Anything above it is a matter of degree and belongs in the report as a number, not in a gate.

⚠️ **Timing.** The embeddings on disk predate the preprocessing corrections and R1 re-embeds, so a
number computed before R1 is indicative, not final. The *shape* of the answer — spread versus collapsed
— is unlikely to flip.

### 2.2 · Stage 7 — the synthetic positive control

**Construction.** For each panel drug, take cell-line pairs whose measured responses differ by an amount
in the **bottom quartile** of that drug's pairwise `|y_A − y_B|`, mix their cells at weight **0.5**, and
label the bag with the mixture-weighted response. Inside such a bag the per-cell ground truth is known
by construction: A-cells should score near `y_A`, B-cells near `y_B`.

**Why the pairs are chosen to be close, not far apart (Selin, 13.08.2026).** The only response
differences this project ever measured are *between* cell lines, so any manufactured control inherits a
between-line magnitude — which is larger than any plausible within-line heterogeneity. Pairing on a
large gap gives a control that is easy to pass and useless as evidence about the regime Q2 actually
operates in. The bottom quartile brings the planted difference down toward that regime. Its cost, taken
deliberately: the control is harder to pass, and a stage-7 failure ends the run — so the run can be
ended by a control tuned finer than the labels can support.

**Quantity: within-bag rank separation, measured as AUROC.** Take "this cell came from line A" as the
group label and the model's predicted response as the score. AUROC is then the probability that a
randomly drawn A-cell is predicted higher than a randomly drawn B-cell — 0.5 is no separation, 1.0 is
perfect. Nothing is being classified; this is a rank statistic, and it is exactly the **Mann–Whitney U**
statistic divided by `n_A · n_B`.

**Pass condition: it beats its permutation null** — shuffle which cells came from which source line,
within the bag, and recompute. That null *is* the Mann–Whitney null, so stage 7 is an ordinary Wilcoxon
rank-sum test and needs no threshold chosen by judgement. **The AUROC itself is reported as the
instrument's measured sensitivity**, and every negative result downstream is stated with it attached.

**Why ordering rather than recovered magnitude (Selin, 13.08.2026).** The alternative was the recovered
gap fraction, `(mean prediction on A-cells − mean prediction on B-cells) / (y_A − y_B)`, which is on the
label's own scale and directly interpretable. It was rejected on two grounds. It is
**calibration-sensitive**, and mean pooling gives the model a standing shrinkage incentive, so a model
that orders cells correctly but pulls them toward the bag mean scores low — meaning stage 7 could fail
for the same reason stage 1 would, which costs stage 7 its independence from the stage it licenses. And
under bottom-quartile pairing its denominator `y_A − y_B` is small by construction, so the ratio is
noisy exactly where it is being used. AUROC is immune to shrinkage because it reads only order, which is
what stages 1 and 2 rest on.

The cost, stated: AUROC carries no scale. It says the model separated the groups, not by how much. The
recovered gap fraction is therefore still **computed and reported** beside it — as a description, not a
gate.

### 2.3 · Stage 1 — spread, against `4a` rather than against a number

**Pass condition: MIL's within-line standard deviation of per-cell predictions exceeds `4a`'s**, on the
same cell lines, the same drugs, the same folds and the same representation. No margin, no fraction.

**Why no threshold is needed.** For one line with per-cell predictions `p_c` and its single label `y`:

```
(1/n) Σ_c (p_c − y)²   =   (p̄ − y)²   +   Var_c(p)
   4a's loss, regrouped     MIL's loss     the term MIL deletes
```

an identity, not an approximation
([Step 03](../docs/steps/03-model-and-training-design.md#the-penalty-on-within-line-variation-is-exact-not-figurative-13082026)).
`4a`'s objective charges for within-line variance at full weight in every batch; MIL's mean-pooled
objective does not contain the term at all. So `4a`'s within-line spread is spread that survived an
explicit penalty, and MIL's is spread with that penalty removed. **Comparing the two tests precisely the
deleted term** — which is why an invented margin would add nothing except a number to defend.

⚠️ **This makes `4a` a dependency of `4b`.** Stage 1 cannot be computed until `4a` has run and written
its per-cell predictions, on matching folds, target and representation. That ordering is now a hard edge
in the R-sequence and did not exist before 13.08.2026.

⚠️ **Do not read `4a`'s existing `pred_std` for this.** The column of that name in
`outputs/panel/panel_per_drug_correlation.csv` is the spread of *line-level* predictions **across** cell
lines — a between-line quantity, and the opposite of what stage 1 needs.

### 2.4 · Stage 2 — reproducibility, the actual test

**Pass condition: per-cell agreement across independent seeds beats the shuffled-cell null** — the same
agreement statistic recomputed after permuting cell identities within each line, which destroys any
cell-specific signal while preserving the marginal distribution of predictions. **Three seeds**
(13.08.2026), which is the minimum that makes "the same cells, under different initializations" a
comparison; more seeds only sharpen the estimate and change no bar.

**The last invented number was removed here too.** An earlier draft required the agreement to reach half
of what stage 7 produced. That fraction was dropped on 13.08.2026 (Selin) on the same grounds as the
others: the shuffled-cell null already separates signal from noise, and a magnitude on top of it would
have been chosen rather than derived. The agreement value is **reported**, so a result that is
statistically clear but small is visible as such rather than hidden behind a pass.

### 2.5 · Stage 6 — the confound veto

Regress each cell's predicted response on total counts, genes detected, mitochondrial fraction and
cell-cycle score, **within line**. The veto fires when the confounds explain the within-line variation —
in which case stages 1 and 2 have measured a sequencing artifact that happens to reproduce across seeds,
because the confounds themselves reproduce across seeds.

### 2.6 · Run scope

| | | |
|---|---|---|
| **Representations** | `pca` and `scgpt`, both, separately | Q2 is asked of each; a split answer locates the structure in the encoding |
| **Loss / weighting** | `alpha = 0.5` only, `4a`'s default | the architecture is the change under test; sweeping `alpha` as well would make a difference unattributable ([the governing rule](../docs/TODO.md)) |
| **Seeds** | 3 | stage 2 needs independent initializations to compare |
| **Bags** | one bag = one cell line, **full-line**, lines weighted equally | [Step 03](../docs/steps/03-model-and-training-design.md#mil--the-bag-model-4b_mil_training-design-fixed-13082026) |

Two runs of three seeds. `4a` must have run first.


## 3 · Closing analysis — what kind of cells were they?

Short and descriptive, and **it gates nothing**. By the time it runs, §2 has already decided whether Q2
is positive. This section says *what was found*, not *whether* something was found — the distinction
matters, because an enrichment discovered here cannot be promoted into evidence afterwards.

Two figures and a table:

**a · What the predictions track (stage 6, reported rather than vetoing).** The same regression the veto
uses, shown rather than thresholded: how much of the within-line variation in per-cell predictions is
explained by each of total counts, genes detected, mitochondrial fraction and cell-cycle score. If the
veto passed, these are all small, and showing them is what makes that credible.

**b · Which annotated programs the predictions track (stage 3).** Kinker et al. 2020 annotated recurrent
heterogeneity programs for this exact dataset, independently of any drug-response label. Both sides are
continuous — each cell has a program score, and instance-level MIL gives each cell a predicted response —
so **correlate the two across a line's cells**, per program, against a within-line permutation null.

*No top-k, decided 12.08.2026 (Selin).* An earlier draft took the most- and least-resistant predicted
cells and tested them for enrichment. Correlating the full continuous signal is strictly better here: it
needs no `k` to justify, uses every cell instead of a slice, and is the same method as (a) above — so the
confound check and the biology check are read on one scale rather than two.

⚠️ **Read the enrichment carefully.** Cell-cycle enrichment is close to guaranteed and would be weak
evidence of anything: the project already refuted *the cell-line effect is largely proliferation*
([Corrections](../docs/steps/corrections-and-dead-ends.md#the-cell-line-effect-is-largely-proliferation)),
and Kinker's two named associations are recorded as **not transferring to this task**
([Dead ends](../docs/steps/corrections-and-dead-ends.md#kinkers-two-named-associations-do-not-transfer-to-this-task)).
A program *other* than cell cycle would be the interesting outcome.

**c · One table** — per drug: does the cell ordering repeat across drugs, or is it drug-specific? A
general axis and a drug-specific subpopulation are different findings, and the table is the cheapest
way to tell them apart.

---

### The criterion is closed (13.08.2026)

**No stage may be added to §2, and none may be dropped from it, once a run exists.** Building the model
first and choosing the criterion afterwards is the failure mode this notebook was written backwards to
prevent — which is why §2 was fixed in prose before any cell below it existed.

That applies to the aggregator in particular. If stage 7 fails, mean pooling is **not** swapped for
something that passes: *fail → change the instrument → retry* is a forking path moved down one level,
and it would leave any subsequent positive unattributable. A stage-7 failure ends the run and means
**Q2 unanswered, the instrument was not demonstrated** — never "no heterogeneity found".